# LeetCode 3213 — Construct String with Minimum Cost bằng Python

- Link: https://leetcode.com/problems/construct-string-with-minimum-cost/
- Độ khó: **Hard**
- Chủ đề: **DP trên tiền tố** (`dp[j]` = chi phí nhỏ nhất dựng `target[:j]`) + tìm mọi từ kết thúc tại `j` thật nhanh: **Aho–Corasick với dictionary link** hoặc **rolling hash theo các độ dài khác nhau**; nhận xét then chốt: tổng độ dài `≤ 5·10⁴` ⇒ chỉ có `O(√S)` độ dài khác nhau

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumCost(self, target: str, words: List[str], costs: List[int]) -> int:
```

## 1. Mục tiêu học

1. Viết DP "ghép xâu" kiểu Word Break: `dp[j] = min_{w kết thúc tại j} dp[j − |w|] + cost(w)`.
2. Gộp từ trùng lặp, giữ chi phí **nhỏ nhất**.
3. Vì `Σ|w| ≤ S` nên số **độ dài khác nhau** `≤ √(2S) ≈ 316` ⇒ tại mỗi vị trí chỉ có `O(√S)` ứng viên.
4. Liệt kê nhanh các từ kết thúc tại `j`: rolling hash theo từng độ dài, hoặc Aho–Corasick đi theo **dictionary (output) link** — mỗi bước trên chuỗi output link là một độ dài khác nhau.
5. Kiểm chứng bằng DP thô `startswith`.

## 2. Đề bài & ví dụ

Cho xâu `target`, mảng xâu `words` và mảng `costs` cùng độ dài. Bắt đầu với `s` rỗng; mỗi thao tác chọn `i`, nối `words[i]` vào cuối `s` với chi phí `costs[i]`. Tìm chi phí nhỏ nhất để `s == target`, hoặc `-1` nếu không thể.

**Ví dụ 1:** `target = "abcdef"`, `words = ["abdef","abc","d","def","ef"]`, `costs = [100,1,1,10,5]` → `7` (`"abc" + "d" + "ef"`).

**Ví dụ 2:** `target = "aaaa"`, `words = ["z","zz","zzz"]`, `costs = [1,10,100]` → `-1`.

**Ràng buộc:** `1 ≤ |target| ≤ 5·10⁴`; `1 ≤ words.length = costs.length ≤ 5·10⁴`; `1 ≤ |words[i]| ≤ |target|`; `Σ|words[i]| ≤ 5·10⁴`; chữ thường; `1 ≤ costs[i] ≤ 10⁴`.

In [1]:
from typing import List
from collections import deque
import random, time

INF = float('inf')
examples = [("abcdef", ["abdef", "abc", "d", "def", "ef"], [100, 1, 1, 10, 5], 7),
            ("aaaa", ["z", "zz", "zzz"], [1, 10, 100], -1)]
for t, w, c, exp in examples:
    print(repr(t), w, c, '-> kỳ vọng', exp)

'abcdef' ['abdef', 'abc', 'd', 'def', 'ef'] [100, 1, 1, 10, 5] -> kỳ vọng 7
'aaaa' ['z', 'zz', 'zzz'] [1, 10, 100] -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. DP ghép xâu (Word Break có trọng số)

`dp[0] = 0`; `dp[j]` = chi phí nhỏ nhất để dựng đúng `target[:j]`. Vì mỗi thao tác **nối vào cuối**, chuỗi thao tác ⇔ một cách chia `target` thành các khúc, mỗi khúc là một từ. Đáp án `dp[n]`.

### 3.2. Số độ dài khác nhau là `O(√S)`

Nếu có `k` độ dài khác nhau thì tổng độ dài `≥ 1 + 2 + … + k = k(k+1)/2`. Với `S = 5·10⁴` ⇒ `k ≤ 315`.

In [2]:
S = 5 * 10 ** 4
k = 0
while (k + 1) * (k + 2) // 2 <= S:
    k += 1
print('số độ dài khác nhau tối đa khi tổng độ dài ≤ 5e4:', k)

số độ dài khác nhau tối đa khi tổng độ dài ≤ 5e4: 315


### 3.3. Aho–Corasick và dictionary link

Automaton Aho–Corasick trên tập từ: sau khi đọc `target[:j]`, trạng thái `u` là **hậu tố dài nhất** của `target[:j]` còn là tiền tố của một từ nào đó. Mọi từ kết thúc tại `j` là các nút trên chuỗi fail-link từ `u` mà **là từ**. **Dictionary link** `out[u]` nhảy thẳng tới nút-từ gần nhất trên chuỗi fail ⇒ chỉ thăm các nút-từ, mỗi nút một độ dài khác nhau ⇒ `O(√S)` bước mỗi vị trí.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `dp = [0, ∞, ∞, 1 ("abc"), 2 (+"d"), ∞, 7 (+"ef")]`; "abdef" không khớp ở vị trí 0 ("abc…" ≠ "abd…"); "def" ở vị trí 3 cho `dp[6] ≤ 1 + 10 = 11`, nhưng `2 + 5 = 7` tốt hơn.

Khó khăn chỉ là **tốc độ** liệt kê các từ khớp: so khớp ngây thơ mỗi vị trí với mọi từ tốn `O(n · S)` = `2.5·10⁹`.

## 5. Cách 1 — Brute Force: DP với `startswith` mọi từ

In [3]:
def brute(target, words, costs):
    n = len(target); dp = [INF] * (n + 1); dp[0] = 0
    for i in range(n):
        if dp[i] == INF:
            continue
        for w, c in zip(words, costs):
            if target.startswith(w, i) and dp[i] + c < dp[i + len(w)]:
                dp[i + len(w)] = dp[i] + c
    return -1 if dp[n] == INF else dp[n]

for t, w, c, exp in examples:
    print('brute:', brute(t, w, c), '| kỳ vọng', exp)

brute: 7 | kỳ vọng 7
brute: -1 | kỳ vọng -1


**Độ phức tạp:** `O(n · Σ|w|)` — oracle cho dữ liệu nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

```
gộp từ trùng → chi phí min
dựng Aho–Corasick, tính fail[], out[] (nút-từ gần nhất trên chuỗi fail)
u = 0
for j = 1..n:
    u = goto(u, target[j-1])
    v = u nếu u là từ, ngược lại out[u]
    dp[j] = min over v trên chuỗi out:  dp[j - len(v)] + cost(v)
```

Tổng `O(Σ|w| · σ + n · √S)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Trie, đi tiến từ mỗi vị trí `i`

In [4]:
def trie_forward(target, words, costs):
    root = {}
    END = '$'
    for w, c in zip(words, costs):
        node = root
        for x in w:
            node = node.setdefault(x, {})
        node[END] = min(node.get(END, INF), c)
    n = len(target); dp = [INF] * (n + 1); dp[0] = 0
    for i in range(n):
        if dp[i] == INF:
            continue
        node = root; j = i
        while j < n and target[j] in node:
            node = node[target[j]]; j += 1
            if END in node and dp[i] + node[END] < dp[j]:
                dp[j] = dp[i] + node[END]
    return -1 if dp[n] == INF else dp[n]

for t, w, c, exp in examples:
    print('trie:', trie_forward(t, w, c), '| kỳ vọng', exp)

trie: 7 | kỳ vọng 7
trie: -1 | kỳ vọng -1


`O(n · maxLen)` — xấu nhất `O(n²)` (một từ rất dài trùng với `target`), quá chậm cho `n = 5·10⁴` nhưng tốt trong thực tế khi từ ngắn.

### 7.2. Cách 3 — Rolling hash theo từng độ dài khác nhau

In [5]:
def rolling_hash(target, words, costs):
    MOD = (1 << 61) - 1; B = 131
    n = len(target)
    P = [1] * (n + 1); Hh = [0] * (n + 1)
    for i, x in enumerate(target):
        Hh[i + 1] = (Hh[i] * B + ord(x)) % MOD
        P[i + 1] = P[i] * B % MOD
    byL = {}
    for w, c in zip(words, costs):
        if len(w) > n:
            continue
        h = 0
        for x in w:
            h = (h * B + ord(x)) % MOD
        d = byL.setdefault(len(w), {})
        if c < d.get(h, INF):
            d[h] = c
    Ls = sorted(byL)
    dp = [INF] * (n + 1); dp[0] = 0
    for j in range(1, n + 1):
        best = INF; hj = Hh[j]
        for L in Ls:
            if L > j:
                break
            c = byL[L].get((hj - Hh[j - L] * P[L]) % MOD)
            if c is not None and dp[j - L] + c < best:
                best = dp[j - L] + c
        dp[j] = best
    return -1 if dp[n] == INF else dp[n]

for t, w, c, exp in examples:
    print('hash:', rolling_hash(t, w, c), '| kỳ vọng', exp)

hash: 7 | kỳ vọng 7
hash: -1 | kỳ vọng -1


`O(n · √S)` phép tra dict. Đụng độ hash với modulo `2^61 − 1` gần như không xảy ra (có thể dùng hai modulo để chắc hơn).

### 7.3. Cách 4 — Aho–Corasick + dictionary link (lời giải chính)

In [6]:
def aho_corasick(target, words, costs):
    best = {}
    for w, c in zip(words, costs):
        if c < best.get(w, INF):
            best[w] = c
    ch = [{}]; cost = [INF]; ln = [0]
    for w, c in best.items():
        u = 0
        for x in w:
            nx = ch[u].get(x)
            if nx is None:
                nx = len(ch); ch[u][x] = nx; ch.append({}); cost.append(INF); ln.append(ln[u] + 1)
            u = nx
        cost[u] = c
    N = len(ch); fail = [0] * N; out = [0] * N
    q = deque(ch[0].values())
    while q:
        u = q.popleft()
        f = fail[u]
        out[u] = f if cost[f] < INF else out[f]        # nút-từ gần nhất trên chuỗi fail (0 = không có)
        for x, v in ch[u].items():
            g = fail[u]
            while g and x not in ch[g]:
                g = fail[g]
            fv = ch[g].get(x, 0)
            fail[v] = fv if fv != v else 0
            q.append(v)
    n = len(target); dp = [INF] * (n + 1); dp[0] = 0; u = 0
    for i, x in enumerate(target):
        while u and x not in ch[u]:
            u = fail[u]
        u = ch[u].get(x, 0)
        v = u if cost[u] < INF else out[u]
        b = INF; j = i + 1
        while v:
            p = dp[j - ln[v]] + cost[v]
            if p < b:
                b = p
            v = out[v]
        dp[j] = b
    return -1 if dp[n] == INF else dp[n]

for t, w, c, exp in examples:
    print('Aho-Corasick:', aho_corasick(t, w, c), '| kỳ vọng', exp)

Aho-Corasick: 7 | kỳ vọng 7
Aho-Corasick: -1 | kỳ vọng -1


**Lưu ý thứ tự BFS:** `out[u]` dùng `fail[u]` và `out[fail[u]]`; nút `fail[u]` nông hơn `u` nên đã được xử lý (BFS theo độ sâu) — ta tính `out[u]` ngay khi lấy `u` khỏi hàng đợi, lúc đó `fail[u]` đã gán (khi duyệt cha của `u`).

## 8. TRACE từng bước

Trace Aho–Corasick trên ví dụ 1: ở mỗi vị trí in trạng thái (xâu của nút) và các từ tìm được qua dictionary link.

In [7]:
def trace(target, words, costs):
    best = {}
    for w, c in zip(words, costs):
        best[w] = min(best.get(w, INF), c)
    ch = [{}]; cost = [INF]; ln = [0]; label = ['']
    for w, c in best.items():
        u = 0
        for x in w:
            if x not in ch[u]:
                ch[u][x] = len(ch); ch.append({}); cost.append(INF); ln.append(ln[u] + 1); label.append(label[u] + x)
            u = ch[u][x]
        cost[u] = c
    N = len(ch); fail = [0] * N; out = [0] * N; q = deque(ch[0].values())
    while q:
        u = q.popleft(); f = fail[u]; out[u] = f if cost[f] < INF else out[f]
        for x, v in ch[u].items():
            g = fail[u]
            while g and x not in ch[g]:
                g = fail[g]
            fv = ch[g].get(x, 0); fail[v] = fv if fv != v else 0; q.append(v)
    dp = [INF] * (len(target) + 1); dp[0] = 0; u = 0
    for i, x in enumerate(target):
        while u and x not in ch[u]:
            u = fail[u]
        u = ch[u].get(x, 0); j = i + 1
        v = u if cost[u] < INF else out[u]; found = []
        while v:
            found.append((label[v], cost[v])); dp[j] = min(dp[j], dp[j - ln[v]] + cost[v]); v = out[v]
        print(f'j = {j}: đọc {x!r}, trạng thái {label[u]!r:8s} | từ kết thúc tại j: {found} | dp[{j}] = {dp[j]}')

trace(*examples[0][:3])

j = 1: đọc 'a', trạng thái 'a'      | từ kết thúc tại j: [] | dp[1] = inf
j = 2: đọc 'b', trạng thái 'ab'     | từ kết thúc tại j: [] | dp[2] = inf
j = 3: đọc 'c', trạng thái 'abc'    | từ kết thúc tại j: [('abc', 1)] | dp[3] = 1
j = 4: đọc 'd', trạng thái 'd'      | từ kết thúc tại j: [('d', 1)] | dp[4] = 2
j = 5: đọc 'e', trạng thái 'de'     | từ kết thúc tại j: [] | dp[5] = inf
j = 6: đọc 'f', trạng thái 'def'    | từ kết thúc tại j: [('def', 10), ('ef', 5)] | dp[6] = 7


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def minimumCost(self, target: str, words: List[str], costs: List[int]) -> int:
        INF = float('inf')
        best = {}
        for w, c in zip(words, costs):
            if c < best.get(w, INF):
                best[w] = c
        ch = [{}]
        cost = [INF]
        ln = [0]
        for w, c in best.items():
            u = 0
            for x in w:
                nx = ch[u].get(x)
                if nx is None:
                    nx = len(ch)
                    ch[u][x] = nx
                    ch.append({})
                    cost.append(INF)
                    ln.append(ln[u] + 1)
                u = nx
            cost[u] = c
        N = len(ch)
        fail = [0] * N
        out = [0] * N
        q = deque(ch[0].values())
        while q:
            u = q.popleft()
            f = fail[u]
            out[u] = f if cost[f] < INF else out[f]
            for x, v in ch[u].items():
                g = fail[u]
                while g and x not in ch[g]:
                    g = fail[g]
                fv = ch[g].get(x, 0)
                fail[v] = fv if fv != v else 0
                q.append(v)
        n = len(target)
        dp = [INF] * (n + 1)
        dp[0] = 0
        u = 0
        for i, x in enumerate(target):
            while u and x not in ch[u]:
                u = fail[u]
            u = ch[u].get(x, 0)
            v = u if cost[u] < INF else out[u]
            b = INF
            j = i + 1
            while v:
                p = dp[j - ln[v]] + cost[v]
                if p < b:
                    b = p
                v = out[v]
            dp[j] = b
        return -1 if dp[n] == INF else dp[n]

for t, w, c, exp in examples:
    got = Solution().minimumCost(t, w, c)
    print(got, 'OK' if got == exp else 'SAI')

7 OK
-1 OK


## 10. Vì sao thuật toán đúng

1. **DP:** chuỗi thao tác nối ⇔ phân hoạch `target` thành các khúc liên tiếp là từ; chi phí cộng tính ⇒ `dp[j] = min dp[j − |w|] + cost(w)` trên mọi từ `w` là hậu tố của `target[:j]`.
2. **Aho–Corasick:** sau khi đọc `target[:j]`, các hậu tố của `target[:j]` là tiền tố của một từ chính là các nút trên chuỗi fail từ `u`. Từ `w` là hậu tố của `target[:j]` ⇒ `w` là một nút trên chuỗi đó. Dictionary link bỏ qua các nút không phải từ ⇒ duyệt **đúng và đủ** các từ kết thúc tại `j`.
3. **Độ phức tạp:** các nút-từ trên chuỗi có độ dài giảm ngặt và đều là độ dài của từ ⇒ `≤ √(2S)` bước.
4. Gộp từ trùng lấy min không làm mất nghiệm (dùng từ đắt hơn không bao giờ có lợi).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DP `startswith` | `O(n · S)` | `O(n)` |
| 2. Trie tiến | `O(S + n · maxLen)` | `O(S)` |
| 3. Rolling hash | `O(S + n · √S)` | `O(n + S)` |
| 4. Aho–Corasick | `O(S · σ + n · √S)` (`σ`: vòng `while` fail, khấu hao) | `O(S)` |

`n = S = 5·10⁴`: `n · √S ≈ 1.6·10⁷` trong ca tệ nhất — ca tệ nhất của cách 4 thực tế chỉ ~1–2 s trong Python.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'trie': trie_forward, 'hash': rolling_hash, 'aho': aho_corasick,
         'submit': lambda t, w, c: Solution().minimumCost(t, w, c)}
edge = [("a", ["a"], [5], 5), ("a", ["b"], [1], -1), ("aaa", ["a", "aa"], [3, 4], 7),
        ("ab", ["a", "b", "ab"], [1, 1, 5], 2), ("abc", ["abc", "abc"], [9, 4], 4),
        ("abab", ["ab", "ba", "a", "b"], [1, 1, 10, 10], 2), ("abc", ["abcd", "ab"], [1, 1], -1)]
for t, w, c, exp in examples + edge:
    assert brute(t, w, c) == exp, (t, w, c, brute(t, w, c))
    for name, f in impls.items():
        assert f(t, w, c) == exp, (name, t, w, c)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với brute

In [10]:
rng = random.Random(3213)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 12); t = ''.join(rng.choice('ab') for _ in range(n))
    k = rng.randint(1, 7)
    w = [''.join(rng.choice('ab') for _ in range(rng.randint(1, min(n, 4)))) for _ in range(k)]
    if rng.random() < 0.5:
        p = rng.randint(0, n - 1); w.append(t[p:p + rng.randint(1, 3)])
    if rng.random() < 0.3:
        w.append(rng.choice(w))                        # từ trùng lặp
    c = [rng.randint(1, 9) for _ in w]
    exp = brute(t, w, c)
    for name, f in impls.items():
        assert f(t, w, c) == exp, (name, t, w, c)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 1500 bộ ngẫu nhiên.


### 12.3. Ca lớn (`n = 5·10⁴`, `Σ|w| ≤ 5·10⁴`)

In [11]:
n = 5 * 10 ** 4
cases = {}
ws = ['a' * L for L in range(1, 316)]                                   # 315 độ dài khác nhau, Σ ≈ 4.98e4
cases['toàn a, 315 độ dài'] = ('a' * n, ws, [rng.randint(1, 10 ** 4) for _ in ws])
t = ''.join(rng.choice('abc') for _ in range(n))
ws = []; tot = 0
while tot < n - 20:
    p = rng.randint(0, n - 10); L = rng.randint(1, 8); ws.append(t[p:p + L]); tot += len(ws[-1])
ws += list('abc')
cases['ngẫu nhiên abc'] = (t, ws, [rng.randint(1, 10 ** 4) for _ in ws])
cases['một từ dài'] = ('ab' * (n // 2), ['ab' * (n // 4), 'ab'], [7, 1])
for name, (t, w, c) in cases.items():
    tm = time.time(); x = Solution().minimumCost(t, w, c); t1 = time.time() - tm
    print(f'{name:20s}: {x:10d} | nộp {t1:.2f}s')
    if name != 'toàn a, 315 độ dài':
        assert x == rolling_hash(t, w, c)
assert Solution().minimumCost('ab' * (n // 2), ['ab' * (n // 4), 'ab'], [7, 1]) == 14
print('Một từ dài: dùng 2 lần từ dài (7 + 7) rẻ hơn 25000 lần "ab": đúng.')

toàn a, 315 độ dài  :      36610 | nộp 1.10s
ngẫu nhiên abc      :     356375 | nộp 0.05s


một từ dài          :         14 | nộp 0.03s
Một từ dài: dùng 2 lần từ dài (7 + 7) rẻ hơn 25000 lần "ab": đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam lấy từ **dài nhất** khớp tại vị trí hiện tại

In [12]:
def wrong_longest(target, words, costs):
    i = tot = 0; n = len(target)
    while i < n:
        cand = [(len(w), -c) for w, c in zip(words, costs) if target.startswith(w, i)]
        if not cand:
            return -1
        L, negc = max(cand); tot -= negc; i += L                    # SAI
    return tot

print('sai :', wrong_longest("aaa", ["aa", "a"], [10, 1]), '| đúng:', Solution().minimumCost("aaa", ["aa", "a"], [10, 1]))
print('sai :', wrong_longest("aab", ["aa", "a", "ab"], [1, 1, 1]), '| đúng:', Solution().minimumCost("aab", ["aa", "a", "ab"], [1, 1, 1]))

sai : 11 | đúng: 3
sai : -1 | đúng: 2


**Vì sao sai:** từ dài có thể đắt, hoặc đẩy ta vào ngõ cụt (`"aa"` rồi không có từ `"b"`).

### Lỗi 2 — Tham lam lấy từ **rẻ nhất** khớp tại vị trí hiện tại

In [13]:
def wrong_cheapest(target, words, costs):
    i = tot = 0; n = len(target)
    while i < n:
        cand = [(c, -len(w)) for w, c in zip(words, costs) if target.startswith(w, i)]
        if not cand:
            return -1
        c, negL = min(cand); tot += c; i -= negL                    # SAI
    return tot

args = ("abc", ["a", "ab", "bc", "c"], [1, 1, 1, 100])
print('sai :', wrong_cheapest(*args), '| đúng:', Solution().minimumCost(*args))

sai : 101 | đúng: 2


**Vì sao sai:** chọn `"ab"` (rẻ, dài hơn khi hoà) buộc dùng `"c"` giá 100; tối ưu là `"a" + "bc"` = 2. Bài có cấu trúc con tối ưu nhưng **không** có tính chất tham lam ⇒ cần DP.

### Lỗi 3 — Aho–Corasick chỉ xét **nút hiện tại** (quên dictionary link)

In [14]:
def wrong_no_output_links(target, words, costs):
    best = {}
    for w, c in zip(words, costs):
        best[w] = min(best.get(w, INF), c)
    ch = [{}]; cost = [INF]; ln = [0]
    for w, c in best.items():
        u = 0
        for x in w:
            if x not in ch[u]:
                ch[u][x] = len(ch); ch.append({}); cost.append(INF); ln.append(ln[u] + 1)
            u = ch[u][x]
        cost[u] = c
    fail = [0] * len(ch); q = deque(ch[0].values())
    while q:
        u = q.popleft()
        for x, v in ch[u].items():
            g = fail[u]
            while g and x not in ch[g]:
                g = fail[g]
            fv = ch[g].get(x, 0); fail[v] = fv if fv != v else 0; q.append(v)
    dp = [INF] * (len(target) + 1); dp[0] = 0; u = 0
    for i, x in enumerate(target):
        while u and x not in ch[u]:
            u = fail[u]
        u = ch[u].get(x, 0)
        if cost[u] < INF:                                             # SAI: chỉ từ dài nhất kết thúc tại j
            dp[i + 1] = dp[i + 1 - ln[u]] + cost[u]
    return -1 if dp[-1] == INF else dp[-1]

args = ("ab", ["a", "b", "ab"], [1, 1, 5])
print('sai :', wrong_no_output_links(*args), '| đúng:', Solution().minimumCost(*args))

sai : 5 | đúng: 2


**Vì sao sai:** ở `j = 2` trạng thái là `"ab"`, nhưng `"b"` cũng kết thúc tại đó (là hậu tố) — phải đi theo chuỗi fail/output để thấy.

### Lỗi 4 — Gộp từ trùng bằng `dict(zip(words, costs))` (giữ chi phí **cuối**, không phải nhỏ nhất)

In [15]:
def wrong_overwrite(target, words, costs):
    d = dict(zip(words, costs))                                      # SAI: ghi đè
    return brute(target, list(d), list(d.values()))

args = ("a", ["a", "a"], [1, 5])
print('sai :', wrong_overwrite(*args), '| đúng:', Solution().minimumCost(*args))

sai : 5 | đúng: 1


**Vì sao sai:** `words` có thể chứa cùng một xâu nhiều lần với chi phí khác nhau.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Ghép `target` từ các mảnh, tối thiểu chi phí/số mảnh" ⇒ DP tiền tố kiểu Word Break.
- Ràng buộc **tổng độ dài** từ ⇒ nghĩ ngay "số độ dài khác nhau `O(√S)`".
- Nhiều mẫu cần tìm trong một văn bản ⇒ Aho–Corasick; cần *mọi* mẫu kết thúc tại vị trí ⇒ dictionary link.
- Python: tránh cắt xâu `target[i:i+L]` trong vòng lặp lớn — dùng hash tiền tố hoặc automaton.

## 15. Biến thể của bài

1. **LC 139 Word Break:** chỉ hỏi có ghép được không (`bool`).
2. **LC 140 Word Break II:** liệt kê mọi cách ghép.
3. **LC 3291/3292:** mảnh là **tiền tố** của các từ (không cần cả từ) ⇒ quan sát `dp` không giảm, dùng Z-function/Aho–Corasick.
4. Đếm số cách ghép (bài tập 4); tối thiểu số thao tác trong các phương án rẻ nhất (bài tập 5).

In [16]:
# Biến thể 1: Word Break (bool) = (chi phí với mọi cost = 1) != -1
def word_break(s, words):
    return Solution().minimumCost(s, words, [1] * len(words)) != -1

assert word_break("leetcode", ["leet", "code"]) is True
assert word_break("applepenapple", ["apple", "pen"]) is True
assert word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]) is False
print('Biến thể Word Break: các ví dụ kinh điển của LC 139 đúng.')

Biến thể Word Break: các ví dụ kinh điển của LC 139 đúng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | DP + `startswith` | `O(n·S)` | Oracle |
| 2 | Trie tiến | `O(n·maxLen)` | Chậm khi có từ rất dài |
| 3 | Rolling hash theo độ dài | `O(n·√S)` | Đơn giản, hằng số lớn |
| 4 | Aho–Corasick + dictionary link | `O(n·√S)` | **Nộp bài**, nhanh nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `target = "abab"`, `words = ["ab", "a", "b"]`, `costs = [3, 1, 1]`.

**Bài 2 (tính tay):** `target = "aaaa"`, `words = ["a", "aa", "aaa"]`, `costs = [3, 5, 7]`.

**Bài 3 (điền TODO):** trả về **danh sách từ** (theo thứ tự nối) của một phương án tối ưu, hoặc `None`.

**Bài 4 (tự viết lại):** đếm **số cách** ghép `target` (mỗi chỉ số `i` là một lựa chọn riêng, kể cả từ trùng), mod `10⁹ + 7`.

**Bài 5 (mini challenge):** trong các phương án có chi phí nhỏ nhất, tìm số thao tác **ít nhất**.

In [17]:
def reconstruct_todo(target, words, costs):
    # TODO
    raise NotImplementedError

def count_ways_todo(target, words):
    # TODO
    raise NotImplementedError

def min_ops_among_cheapest_todo(target, words, costs):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `"ab"` giá 3 hay `"a" + "b"` giá 2? (2) Mỗi nửa `"ab"` tốn 2. (3) `4`.

**Bài 2.** (1) Liệt kê phân hoạch của 4: `4 = 1+1+1+1 = 2+2 = 1+3 = 2+1+1`. (2) Chi phí `12, 10, 10, 11`. (3) `10`.

**Bài 3.** (1) Lưu `par[j] = (j − |w|, w)` khi cập nhật `dp[j]`. (2) Truy ngược từ `n`. (3) Đảo danh sách.

**Bài 4.** (1) `ways[j] = Σ ways[j − |w|] · (số lần w xuất hiện trong words)`. (2) Dùng cùng automaton, thay `min` bằng tổng. (3) Kiểm bằng đệ quy vét cạn.

**Bài 5.** (1) Dùng cặp `(cost, ops)` và so sánh từ điển. (2) `dp[j] = min (dp[j−L][0] + c, dp[j−L][1] + 1)`. (3) Kiểm bằng vét cạn mọi phân hoạch.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().minimumCost("abab", ["ab", "a", "b"], [3, 1, 1]) == 4
assert Solution().minimumCost("aaaa", ["a", "aa", "aaa"], [3, 5, 7]) == 10

def reconstruct(target, words, costs):
    n = len(target); dp = [INF] * (n + 1); dp[0] = 0; par = [None] * (n + 1)
    for i in range(n):
        if dp[i] == INF:
            continue
        for w, c in zip(words, costs):
            if target.startswith(w, i) and dp[i] + c < dp[i + len(w)]:
                dp[i + len(w)] = dp[i] + c; par[i + len(w)] = (i, w)
    if dp[n] == INF:
        return None
    res = []; j = n
    while j:
        i, w = par[j]; res.append(w); j = i
    return res[::-1]

for _ in range(300):
    n = rng.randint(1, 10); t = ''.join(rng.choice('ab') for _ in range(n))
    w = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 3))) for _ in range(rng.randint(1, 6))]
    c = [rng.randint(1, 9) for _ in w]
    r = reconstruct(t, w, c); best = Solution().minimumCost(t, w, c)
    if r is None:
        assert best == -1
    else:
        cmin = {x: min(cc for ww, cc in zip(w, c) if ww == x) for x in w}
        assert ''.join(r) == t and sum(cmin[x] for x in r) == best
print('Bài 1: 4 | Bài 2: 10 | Bài 3:', reconstruct(*examples[0][:3]))

Bài 1: 4 | Bài 2: 10 | Bài 3: ['abc', 'd', 'ef']


In [19]:
# Bài 4, 5
MOD = 10 ** 9 + 7
from collections import Counter
from functools import lru_cache

def count_ways(target, words):
    mult = Counter(words)
    byL = {}
    for w, k in mult.items():
        byL.setdefault(len(w), {})[w] = k
    n = len(target); ways = [0] * (n + 1); ways[0] = 1
    for j in range(1, n + 1):
        s = 0
        for L, d in byL.items():
            if L <= j:
                k = d.get(target[j - L:j])
                if k:
                    s += ways[j - L] * k
        ways[j] = s % MOD
    return ways[n]

def min_ops_among_cheapest(target, words, costs):
    n = len(target); dp = [(INF, INF)] * (n + 1); dp[0] = (0, 0)
    for i in range(n):
        if dp[i][0] == INF:
            continue
        for w, c in zip(words, costs):
            if target.startswith(w, i):
                cand = (dp[i][0] + c, dp[i][1] + 1)
                if cand < dp[i + len(w)]:
                    dp[i + len(w)] = cand
    return -1 if dp[n][0] == INF else dp[n][1]

def enum_partitions(target, words, costs):
    res = []
    def go(i, c, k):
        if i == len(target):
            res.append((c, k)); return
        for w, cc in zip(words, costs):
            if target.startswith(w, i):
                go(i + len(w), c + cc, k + 1)
    go(0, 0, 0)
    return res

for _ in range(300):
    n = rng.randint(1, 8); t = ''.join(rng.choice('ab') for _ in range(n))
    w = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 3))) for _ in range(rng.randint(1, 5))]
    c = [rng.randint(1, 4) for _ in w]
    parts = enum_partitions(t, w, c)
    assert count_ways(t, w) == len(parts) % MOD
    exp = -1 if not parts else min(parts)[1]
    assert min_ops_among_cheapest(t, w, c) == exp
print('Bài 4 (đếm cách) và Bài 5 (ít thao tác nhất trong các phương án rẻ nhất) khớp vét cạn.')

Bài 4 (đếm cách) và Bài 5 (ít thao tác nhất trong các phương án rẻ nhất) khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Aho–Corasick: ch, fail, out (nút-từ gần nhất trên chuỗi fail), cost, ln
u = 0
for i, x in enumerate(target):
    while u and x not in ch[u]: u = fail[u]
    u = ch[u].get(x, 0)
    v = u if cost[u] < INF else out[u]
    while v:
        dp[i + 1] = min(dp[i + 1], dp[i + 1 - ln[v]] + cost[v]); v = out[v]
```

**Checklist:**

- [ ] Gộp từ trùng lấy **min** chi phí?
- [ ] Duyệt **mọi** từ kết thúc tại `j` (dictionary link), không chỉ từ dài nhất?
- [ ] `fail` của nút độ sâu 1 là gốc; BFS theo độ sâu để `out` đúng?
- [ ] Trả `-1` khi `dp[n]` vô cực?
- [ ] Tránh cắt xâu trong vòng lặp `O(n · √S)` ở Python?